In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy
import seaborn as sns
from scipy import stats
from scipy.stats import ttest_rel, ttest_1samp
from config import *
from plotting_functions import *
import analysis_helpers as helper
import itertools
sys.path.insert(0,'../..')
import plotting_functions
# reimport magic
import importlib
importlib.reload(helper)
importlib.reload(plotting_functions)

<module 'plotting_functions' from '/Users/elb/Desktop/code_packages_maintainer_version/avatarRT_analysis/plotting_functions.py'>

In [2]:
# a function that subselects points from a distribution such that the probability of selecting a
# point i is inverselt proportional to its distance from j (which by default is the mean of the distribution)
def simulate_subselection(distrib, j=None, proportion=0.5):
    if j is None: j = np.mean(distrib)
    # calculate the distance of each point from j
    distances = np.abs(distrib - j)
    # calculate the probabilities of selecting each point as inversely proportional to its distance from j
    probabilities = 1 / (distances + 1e-6)  # add a small constant to avoid division by zero
    probabilities /= np.sum(probabilities)  # normalize to sum to 1
    # number of points to select
    num_points_to_select = int(len(distrib) * proportion)
    # randomly select points based on the calculated probabilities
    selected_indices = np.random.choice(len(distrib), size=num_points_to_select, replace=False, p=probabilities)
    deleted_points = distrib[selected_indices]
    # extract the selected points
    distrib = distrib[np.setdiff1d(np.arange(len(distrib)), selected_indices)]
    # if the number of points in distrib is less than the original number of points,
    # upsample the remaining points to maintain the original number of points
    selected_indices = np.random.choice(len(distrib), size=len(distrib)+num_points_to_select, replace=True) 
    distrib = distrib[selected_indices]
    return distrib 

# scale data
def scale_data(X, maxx=1, minn=-1, meann=None):
    if meann != None:
        X = X - np.mean(X) + meann
    X_std = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))
    X_scaled = X_std * (maxx - minn) + minn
    return X_scaled

def simulate_realigned_distribution(initial_distribution, scale_factor=1.5, size=None, bins=50, distributions=None, return_func=False):
    # determine best fit distribution
    dist, params = best_fit_distribution_v2(initial_distribution, bins=bins, distributions=distributions, verbose=False)
    *shapes, loc, scale = params
    #print(loc,scale,dist.name)
    new_scale = scale * scale_factor
    if size is None:
        size = len(initial_distribution)
    # Sample from the distribution with the new scale
    new_data = dist.rvs(*shapes, loc=loc, scale=new_scale, size=size)
    new_data = scale_data(new_data, maxx=np.max(initial_distribution), minn=np.min(initial_distribution), meann=np.mean(initial_distribution))
    if return_func: 
        return new_data, dist
    return new_data


def best_fit_distribution(data, bins=50, distributions=None, verbose=False):
    """
    Find the best fitting distribution to data from a list of candidate distributions.
    Returns the name of the best fit and its parameters.
    """
    if distributions is None:
        # Common continuous distributions to try
        distributions = [
            stats.norm, stats.lognorm, stats.gamma, stats.beta,
            stats.triang
        ]
    data = np.asarray(data)
    y, x = np.histogram(data, bins=bins, density=True)
    x = (x + np.roll(x, -1))[:-1] / 2.0

    best_sse = np.inf
    best_dist = None
    best_params = None

    for distribution in distributions:
        try:
            # Fit distribution to data
            params = distribution.fit(data)
            # Separate parts of parameters
            location, scale = params[-2], params[-1]
            # Get fitted PDF
            pdf = distribution.pdf(x, *params[:-2], loc=location, scale=scale)
            # Compute sum of squared errors
            sse = np.sum(np.square(y - pdf))
            if verbose:
                print(f"{distribution.name}: SSE={sse:.4f}")
            if sse < best_sse:
                best_sse = sse
                best_dist = distribution
                best_params = params
        except Exception as e:
            if verbose:
                print(f"{distribution.name} failed: {e}")
            continue
    if verbose:
        print(f'best distribution: {best_dist.name}')
    return best_dist, best_params

def best_fit_distribution_v2(data, bins=None, distributions=None, verbose=False):
    if distributions is None:
        # Common continuous distributions to try
        distributions = [
            stats.norm, stats.lognorm, stats.gamma, stats.beta,
            stats.triang
        ]
    data = np.asarray(data)
    distests = []
    params = []
    for distribution in distributions:
        args = distribution.fit(data)
        distest = stats.kstest(data, distribution.cdf, args=args)
        if verbose:
            print(f"{distribution.name}: distest={distest}")
        distests.append(distest.statistic)
        params.append(args)
    idx = np.argmin(distests)
    #print(f'best distribution: {distributions[idx].name}')
    return distributions[idx], params[idx]
        
         

def run_simulations(initial_data, proportion=0.5, n_sims=1000):
    sim_subselection = []
    sim_realignment = []
    distribution_type=[]
    for i in range(n_sims):
        sim_subselection.append(simulate_subselection(initial_data, proportion=proportion))
        if len(distribution_type)==0:
            r, dist = simulate_realigned_distribution(initial_data, scale_factor=1+proportion, size=None, bins=50, return_func=True)
            distribution_type.append(dist)
            sim_realignment.append(r)
        else:
            sim_realignment.append(simulate_realigned_distribution(initial_data, scale_factor=1+proportion, size=None, 
                                                                   distributions=distribution_type, bins=50))
    return np.array(sim_subselection), np.array(sim_realignment)

def run_single_simulation(initial_data, proportion=0.5):
    sim_subselection = simulate_subselection(initial_data, proportion=proportion)
    sim_realignment, dis = simulate_realigned_distribution(initial_data, scale_factor=1+proportion, size=None, bins=50, return_func=True)
    print(f'In run single simulation: proportion={proportion}, fit dist {dis.name}')

    # plot
    fig,ax=plt.subplots(1,1,figsize=(8, 6))
    sns.histplot(initial_data, bins=40, alpha=0.2, ax=ax,label='original data', kde=True, stat='proportion', edgecolor=None)
    sns.histplot(sim_subselection, bins=40, alpha=0.2, ax=ax,label='subselected sim', kde=True, stat='proportion', edgecolor=None)
    sns.histplot(sim_realignment, bins=40, alpha=0.2, ax=ax,label='realigned sim', kde=True, stat='proportion', edgecolor=None)
    bin_data = lambda b:np.histogram(b, bins=20, density=True)[0]
    get_ent = lambda b:scipy.stats.entropy(bin_data(b) + 1e-6)
    mystring= f'orig: var={np.var(initial_data):.3f}, ent={get_ent(initial_data):.3f}\nsubsel: var={np.var(sim_subselection):.3f}, ent={get_ent(sim_subselection):.3f}\nrealign: var={np.var(sim_realignment):.3f}, ent={get_ent(sim_realignment):.3f}'
    ax.text(0.01, 0.99, mystring, transform=ax.transAxes,
            ha='left', va='top', fontsize=12)
    
    plt.legend()
    plt.xlabel('decoded angle')
    plt.ylabel('proportion')
    plt.title('Distributions')
    plt.legend()
    sns.despine()
    print(f'finished run single simulation')
    
def compute_change_scores(pre_learning, post_learning):
    pre_var = np.var(pre_learning)
    post_var = np.var(post_learning)
    delta_var = post_var - pre_var
    pre_ent = scipy.stats.entropy(np.histogram(pre_learning, bins=50, density=True)[0] + 1e-6)
    post_ent = scipy.stats.entropy(np.histogram(post_learning, bins=50, density=True)[0] + 1e-6)
    delta_ent = post_ent - pre_ent
    return delta_var, delta_ent

def plot_simulations(subsel_scores, realign_scores, true_scores, outfn=None):
    
    # get x and y limits for the plots
    var_lims = np.max([np.abs(subsel_scores[:,0]).max(), np.abs(realign_scores[:,0]).max(), np.abs(true_scores[0]).max()])
    var_lims = [-var_lims-0.01, var_lims+0.01]
    
    # get the maximum absolute valute of the entropy changes to set the x limits
    entropy_lims = np.max([np.abs(subsel_scores[:,1]).max(), np.abs(realign_scores[:,1]).max(), np.abs(true_scores[1]).max()])
    entropy_lims = [-entropy_lims-0.05, entropy_lims+0.05]

    # plot the distribution of subsel and realign scores for variance and entropy changes
    # with the true score overlaid
    fig, axes = plt.subplots(2,2,figsize=(12,10),sharey=True,sharex=False)
    axes=axes.ravel()
    g=sns.histplot(subsel_scores[:,0], bins=30, color='blue', alpha=0.2, label='subselection variance', stat='count',ax=axes[0])
    axes[0].axvline(true_scores[0], color='red', linestyle='--', label='true variance change')
    g.set(  xlim=var_lims, title=r'$\Delta$ variance subselection')
    subsel_var_z = (true_scores[0] - np.mean(subsel_scores[:,0])) / np.std(subsel_scores[:,0])
    # print the z-score on the plot inside the legend

    g=sns.histplot(realign_scores[:,0], bins=30, color='green', alpha=0.2, label='realignment variance', stat='count',ax=axes[1])
    axes[1].axvline(true_scores[0], color='red', linestyle='--', label='true variance change')
    g.set( xlim=var_lims, title=r'$\Delta$ variance realignment')
    # compute the z-score of the true score relative to the distribution of simulated scores
    realign_var_z = (true_scores[0] - np.mean(realign_scores[:,0])) / np.std(realign_scores[:,0])
    # print the z-score on the plot inside the legend
    g=sns.histplot(subsel_scores[:,1], bins=30, color='blue', alpha=0.2, label='subselection entropy', stat='count',ax=axes[2])
    axes[2].axvline(true_scores[1], color='red', linestyle='--', label='true entropy change')
    g.set(  xlim=entropy_lims, title=r'$\Delta$ entropy subselection')
    # compute the z-score of the true score relative to the distribution of simulated scores
    subsel_ent_z = (true_scores[1] - np.mean(subsel_scores[:,1])) / np.std(subsel_scores[:,1])
    
    g=sns.histplot(realign_scores[:,1], bins=30, color='green', alpha=0.2, label='realignment entropy', stat='count',ax=axes[3])
    axes[3].axvline(true_scores[1], color='red', linestyle='--', label='true entropy change')
    g.set( xlim=entropy_lims, title=r'$\Delta$ entropy realignment')
    # compute the z-score of the true score relative to the distribution of simulated scores
    realign_ent_z = (true_scores[1] - np.mean(realign_scores[:,1])) / np.std(realign_scores[:,1])
    zscores = [subsel_var_z, realign_var_z, subsel_ent_z, realign_ent_z]
    for i, ax in enumerate(axes):
        zi=zscores[i]
        string = f'z={zi:02f}'
        ax.text(0.05,0.95, s=string, transform=ax.transAxes,
                ha='left', va='top', fontsize=12)
        ax.legend()
    if outfn:
        plt.savefig(outfn)
        print(f'saved to {outfn}')
        plt.close()
    

def score_simulations(subselection_sims, realignment_sims, pre_learning, post_learning, make_plot=False, plot_outfn=None):
    subsel_scores = np.array([compute_change_scores(pre_learning, sim) for sim in subselection_sims])
    realign_scores = np.array([compute_change_scores(pre_learning, sim) for sim in realignment_sims])
    true_scores = compute_change_scores(pre_learning, post_learning)
    if make_plot:
        plot_simulations(subsel_scores, realign_scores, true_scores, outfn=plot_outfn)
    return subsel_scores, realign_scores, true_scores

    


In [3]:
def get_session_info(subject_info):
    # loads the 0th index component (the IM)
    im_session = int(subject_info['im_session'].item())
    im_component, im_idx = 0, 0

    # loads the WMP; adjusts for 0 indexing (but that 0 is the IM, so 1=1)
    wmp_session = int(subject_info['wmp_session'].item())
    wmp_component=int(subject_info['wmp_component'].item())
    wmp_idx = wmp_component
    if wmp_component != 1: wmp_idx = wmp_component - 1
    else: wmp_idx = 1

    # loads the OMP; adjusts for 0 indexing (but that 0 is the IM, so 1=1)
    omp_session = int(subject_info['omp_session'].item())
    omp_component=int(subject_info['omp_component'].item())
    omp_idx = omp_component - 1 
    return im_session, wmp_session, omp_session

In [4]:
def compute_statistics(true_score, simulated_distribution):
    z = (true_score- np.mean(simulated_distribution)) / np.std(simulated_distribution)
    pt=scipy.stats.norm.sf(abs(z))*2
    return z,pt

def compute_mse(true_score, simulated_distribution):
    trues = np.repeat(true_score, len(simulated_distribution))
    return np.mean((trues - simulated_distribution)**2)

def convert_z_to_score(z, distribution):
    """Convert a z-score to a value based on the provided distribution."""
    mean = np.mean(distribution)
    std_dev = np.std(distribution)
    orig_value = z * std_dev + mean
    return orig_value

# Compute probability of observing the true value under each distribution
def compute_density(value, distribution):
    kde = scipy.stats.gaussian_kde(distribution, bw_method="scott")
    density = kde(value)
    return density[0]

def compute_normalized_density(value, distribution, minn=None):
    kde = scipy.stats.gaussian_kde(distribution, bw_method='scott')
    d_obs = kde(value)[0]
    d_max = np.max(kde(distribution))
    if minn == None:
        d_min = np.min(kde(distribution))
    else:
        d_min = minn
    d_prop = (d_obs - d_min) / (d_max - d_min)
    return d_prop

def compute_percentile(value, distribution):
    """Compute the percentile of a value within a given distribution."""
    percentile = stats.percentileofscore(distribution, value) + 1e-6  # add small constant to avoid exact 0 or 100
    return np.abs(percentile - 50)/50  # return distance from median (50th percentile)

In [5]:
import time

cumulative_info = helper.load_info_file()
# main_results = pd.read_csv('results/final_results/evr_bc_final.csv')

In [ ]:
for prop in [0.2,0.3,0.4,0.5]:
    matches = 0
    mismatches = 0

    for sub in SUB_IDS:
        if sub in ['avatarRT_sub_09','avatarRT_sub_20']:
            continue
        subject_info = cumulative_info[cumulative_info['subject_ID'] == sub ]
        im_session, wmp_session, omp_session = get_session_info(subject_info)
        
        for ses_type,ses_num in zip(['IM','WMP'],[im_session,wmp_session]):
            t0 = time.time()
            # pltfn=f'plots/data/{sub}_{ses_type}_simulated_distributions.png'
            if ses_type=="IM": first_run=1
            else: first_run=2
            final_run=4
            
            # get the true inital and final distributions
            x0 = helper.get_realtime_outdata(sub , f"ses_0{ses_num}", first_run, data_type='decoded_angles')
            x1 = helper.get_realtime_outdata(sub , f"ses_0{ses_num}", final_run, data_type='decoded_angles')
            x0 = x0[x0==x0]
            x1 = x1[x1==x1]

            B = best_fit_distribution(x0)
            A = best_fit_distribution_v2(x0)

            C = best_fit_distribution(x1)
            D = best_fit_distribution_v2(x1)
            print(A[0].name, B[0].name)
            print(C[0].name, D[0].name)
            if A[0].name == B[0].name:
                matches += 1
            else:
                mismatches += 1
            
            if C[0].name == D[0].name:
                matches += 1
            else:
                mismatches += 1
    print(f'matches = {matches}, mismatches = {mismatches}')

best distribution: beta
best distribution: beta
beta triang
beta beta
best distribution: beta
best distribution: beta
beta beta
beta beta
best distribution: norm
best distribution: lognorm
norm norm
lognorm lognorm
best distribution: beta
best distribution: gamma
beta beta
lognorm gamma
best distribution: beta
best distribution: norm
beta beta
norm norm
best distribution: gamma
best distribution: triang
gamma lognorm
triang triang
best distribution: beta
best distribution: beta
beta beta
beta beta
best distribution: beta
best distribution: gamma
beta beta
gamma gamma
best distribution: triang
best distribution: beta
triang triang
lognorm beta
best distribution: gamma
best distribution: lognorm
gamma triang
triang lognorm
best distribution: beta
best distribution: beta
beta gamma
beta beta
best distribution: beta
best distribution: gamma
beta beta
gamma gamma
best distribution: beta
best distribution: norm
beta lognorm
norm norm
best distribution: beta
best distribution: beta
beta beta


In [6]:
sim_dfs = []
summary_dfs = []
cumulative_info = helper.load_info_file()
nperm=100
for prop in [0.2,0.3,0.4,0.5]:
    for sub in SUB_IDS:
        if sub in ['avatarRT_sub_09','avatarRT_sub_20']:
            continue
        subject_info = cumulative_info[cumulative_info['subject_ID'] == sub ]
        im_session, wmp_session, omp_session = get_session_info(subject_info)

        for ses_type,ses_num in zip(['IM','WMP'],[im_session,wmp_session]):
            t0 = time.time()
            # pltfn=f'plots/data/{sub}_{ses_type}_simulated_distributions.png'
            if ses_type=="IM": first_run=1
            else: first_run=2
            final_run=4
            # if prop is 'BC':
            #     this_prop  = main_results[(main_results['subject_id']==sub)&(main_results['session_type']==ses_type)]['delta_BC'].item()
            #     this_prop = max((this_prop, 0.1))
            # else:
            this_prop=prop
            # get the true inital and final distributions
            x0 = helper.get_realtime_outdata(sub , f"ses_0{ses_num}", first_run, data_type='decoded_angles')
            x1 = helper.get_realtime_outdata(sub , f"ses_0{ses_num}", final_run, data_type='decoded_angles')
            x0 = x0[x0==x0]
            x1 = x1[x1==x1]
             
            subselection_sims, realignment_sims = run_simulations(x0, proportion=this_prop, n_sims=nperm)
            subsel_scores, realign_scores, true_scores = score_simulations(subselection_sims, realignment_sims, 
                                                                           x0, x1, make_plot=False, plot_outfn=None)

            temp = pd.DataFrame({'subselection_simulation_delta_variance':subsel_scores[:,0],
                                 'realignment_simulation_delta_variance':realign_scores[:,0], 
                                 'subselection_simulation_delta_entropy':subsel_scores[:,1],
                                 'realignment_simulation_delta_entropy':realign_scores[:,1]})
            temp['subject_id']=sub
            temp['session_type']=ses_type
            temp['proportion']=prop
            sim_dfs.append(temp)

            # Run some basic scoring
            names = ['subselection','realignment']
            metrics = ['variance','entropy']
            distributions = [subsel_scores, realign_scores]
            temp = {'subject_id':[sub]*4,'session_type':[ses_type]*4,'proportion':[prop]*4,
                     "MSE":[], 'true_score':[],'instantaneous_density':[],'proportional_density':[],'max_normed_density':[],
                    'z':[], 'pval_2t':[], 'metric':[], 'simulated_change':[], 'percentile':[], 'absolute_z':[]}
            for idx0, idx1 in itertools.product([0,1],[0, 1]):
                these_scores=distributions[idx0][:,idx1] 
                name=names[idx0]
                met=metrics[idx1]
                true_score=true_scores[idx1]
                
                z,pv= compute_statistics(true_score, these_scores)
                mse = compute_mse(true_score, these_scores)
                idd = compute_density(true_score, these_scores)
                nd = compute_normalized_density(true_score, these_scores)
                perc = compute_percentile(true_score, these_scores)
                
                temp['MSE'].append(mse)
                temp['true_score'].append(true_score)
                temp['instantaneous_density'].append(idd)
                temp['proportional_density'].append(nd)
                temp['z'].append(z)
                temp['pval_2t'].append(pv)
                temp['metric'].append(met)
                temp['simulated_change'].append(name)
                temp['percentile'].append(perc)
                temp['absolute_z'].append(np.abs(z))
                temp['max_normed_density'].append(compute_normalized_density(true_score, these_scores, 0))
            summary_dfs.append(pd.DataFrame(temp))
            t1=time.time()
            print(f'finished {sub},{ses_type},{prop} in {np.round(t1-t0, 5)}s')
summary_df=pd.concat(summary_dfs)
sim_df=pd.concat(sim_dfs)
# summary_df.to_csv('summary_results_simulations_all_final1.csv')
# sim_df.to_csv('simulations_all_final1.csv')

finished avatarRT_sub_05,IM,0.2 in 0.85852s
finished avatarRT_sub_05,WMP,0.2 in 1.02143s
finished avatarRT_sub_06,IM,0.2 in 0.18029s


/Users/elb/miniconda3/envs/rtcloud_av1/lib/python3.7/site-packages/scipy/stats/_continuous_distns.py:639: RuntimeWarning: invalid value encountered in sqrt
  sk = 2*(b-a)*np.sqrt(a + b + 1) / (a + b + 2) / np.sqrt(a*b)


finished avatarRT_sub_06,WMP,0.2 in 1.05842s
finished avatarRT_sub_07,IM,0.2 in 1.10755s
finished avatarRT_sub_07,WMP,0.2 in 0.84709s


/Users/elb/miniconda3/envs/rtcloud_av1/lib/python3.7/site-packages/scipy/optimize/minpack.py:175: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  warnings.warn(msg, RuntimeWarning)


finished avatarRT_sub_08,IM,0.2 in 1.67722s
finished avatarRT_sub_08,WMP,0.2 in 1.11608s
finished avatarRT_sub_10,IM,0.2 in 1.98846s
finished avatarRT_sub_10,WMP,0.2 in 0.85261s


/Users/elb/miniconda3/envs/rtcloud_av1/lib/python3.7/site-packages/scipy/optimize/minpack.py:175: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last five Jacobian evaluations.
  warnings.warn(msg, RuntimeWarning)


finished avatarRT_sub_11,IM,0.2 in 3.55143s
finished avatarRT_sub_11,WMP,0.2 in 3.56855s
finished avatarRT_sub_13,IM,0.2 in 2.60892s
finished avatarRT_sub_13,WMP,0.2 in 3.26378s
finished avatarRT_sub_14,IM,0.2 in 1.80888s
finished avatarRT_sub_14,WMP,0.2 in 0.90476s
finished avatarRT_sub_15,IM,0.2 in 1.00342s
finished avatarRT_sub_15,WMP,0.2 in 2.08472s
finished avatarRT_sub_16,IM,0.2 in 0.18928s
finished avatarRT_sub_16,WMP,0.2 in 2.09888s
finished avatarRT_sub_17,IM,0.2 in 1.697s
finished avatarRT_sub_17,WMP,0.2 in 0.22327s
finished avatarRT_sub_18,IM,0.2 in 1.11748s
finished avatarRT_sub_18,WMP,0.2 in 1.83435s
finished avatarRT_sub_19,IM,0.2 in 2.16944s
finished avatarRT_sub_19,WMP,0.2 in 0.19591s
finished avatarRT_sub_21,IM,0.2 in 1.00463s
finished avatarRT_sub_21,WMP,0.2 in 0.16108s
finished avatarRT_sub_22,IM,0.2 in 1.7785s
finished avatarRT_sub_22,WMP,0.2 in 1.34014s
finished avatarRT_sub_23,IM,0.2 in 1.31628s
finished avatarRT_sub_23,WMP,0.2 in 2.20492s
finished avatarRT_sub_24

In [7]:
summary_df = pd.read_csv('results/intermediate_results/summary_results_simulations_all_final1.csv')
sim_df= pd.read_csv('results/intermediate_results/simulations_all_final1.csv')

In [ ]:
def plot_results(dataframe, yval, xval, order, hue, hue_order=None, title=None, ylabel=None, xlabel=None, ylim=None, colors=['#1f77b4', '#ff7f0e']):
    """Plot results with barplot and overlaid stripplot."""
    fig, ax = plt.subplots(figsize=(10,6))
    sns.barplot(data=dataframe, x=xval, y=yval, order=order, hue=hue, hue_order=hue_order, ax=ax, alpha=0.5, edgecolor='black', 
                 palette=colors)
    sns.stripplot(data=dataframe, x=xval, y=yval, hue=hue, hue_order=hue_order, palette=colors,
                  dodge=True, ax=ax, alpha=1, edgecolor='black', linewidth=0.5, size=4)
    
    # connect the paired points with a line
    for subject in dataframe['subject_id'].unique():
        for i, proportion in enumerate(order):
            sub_df = dataframe[(dataframe['subject_id']==subject) & (dataframe[xval]==proportion)]
            if len(sub_df)==2:
                ax.plot([i-0.2, i+0.2], sub_df[yval], color='gray', alpha=0.5)
    
    # remove the duplicate legend entries
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), title='Model', loc='upper left')
    # add y axis line at 0
    ax.axhline(0, color='black', linestyle='--')
    if ylim:
        ax.set_ylim(ylim)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel(xlabel)
    
    # compute paired t-test between the two simulation types for each proportion and add the p-value to the plot
    from scipy.stats import ttest_rel
    proportions = sorted(dataframe['proportion'].unique())
    for i, proportion in enumerate(proportions):    
        sub_df = dataframe[dataframe['proportion']==proportion]
        group1 = sub_df[sub_df['simulated_change']=='subselection'][yval]
        group2 = sub_df[sub_df['simulated_change']=='realignment'][yval]
        if len(group1)>1 and len(group2)>1:
            tstat, pvalue = ttest_rel(group1, group2, alternative='less')
            print(f'Proportion {proportion}: p-value={pvalue:.4f}')
            stars = determine_symbol(pvalue)
            ax.text(i, ax.get_ylim()[1]*0.95, stars, ha='center', va='top', fontsize=12)
            # add a line below the stars
            ax.plot([i-0.2, i+0.2], [ax.get_ylim()[1]*0.9, ax.get_ylim()[1]*0.9], color='black', linewidth=1)

    # run t-test relative to zero for each simulation type and add the stars to the plot
    for i, proportion in enumerate(proportions):    
        sub_df = dataframe[dataframe['proportion']==proportion]
        for sim_change in ['subselection','realignment']:
            group = sub_df[sub_df['simulated_change']==sim_change][yval]
            if len(group)>1:
                _, pvalue = ttest_1samp(group, popmean=0, alternative='greater')
                print(f'Proportion {proportion}, {sim_change}: p-value={pvalue:.4f}')
                stars = determine_symbol(pvalue)
                # position the stars above the bars
                if sim_change=='subselection':
                    x_pos = i - 0.2
                else:
                    x_pos = i + 0.2
                ax.text(x_pos, ax.get_ylim()[1]*0.85, stars, ha='center', va='top', fontsize=10)


In [13]:
summary_df = pd.read_csv('results/final_results/summary_results_simulations_all_final1.csv')

In [ ]:
colors = sns.color_palette("Set2", 3)
fig,axes=plt.subplots(1,2, figsize=(12,4), sharex=True, sharey=True)
df_sum=summary_df
proportions=df_sum.proportion.unique()
for ax, ses in zip(axes, ['IM','WMP']):
    df_temp = df_sum[(df_sum['metric']=='variance') & (df_sum['session_type'].isin([ses]))].reset_index(drop=True)
    g=sns.barplot(data=df_temp, x='proportion', y='max_normed_density', hue='simulated_change', hue_order=['subselection','realignment'],
                  palette=colors[1:], alpha=0.5, edgecolor='black',  ax=ax, order=proportions)

    sns.stripplot(data=df_temp, x='proportion', y='max_normed_density', hue='simulated_change', hue_order=['subselection','realignment'],  
                         palette=colors[1:], alpha=0.9, edgecolor='k',linewidth=0.5, order=proportions, ax=ax, jitter=False, size=3, dodge=True)

    for i,proportion in enumerate(proportions[:]):
        pair_df = df_temp[df_temp['proportion']==proportion]

        for sub in df_temp['subject_id'].unique():
            sub_df = pair_df[pair_df['subject_id']==sub]
            if len(sub_df)==2:
                ax.plot([i-0.2, i+0.2], sub_df['max_normed_density'], color='gray', alpha=0.5, linewidth=.4)
        # run paired samples t-test between the two simulation types
        group2 = pair_df[pair_df['simulated_change']=='subselection']['max_normed_density']
        group1 = pair_df[pair_df['simulated_change']=='realignment']['max_normed_density']

        if len(group1)>1 and len(group2)>1:
            # Get pvalue from permutation test
            _, p, _ = helper.permutation_test(np.array([group1, group2]), n_iterations=10000, alternative='two-sided')    
            # get effect size as the difference in means between the two groups
            meann, lowerr, upperr, semm = helper.bootstrap_ci(np.array([group1, group2]), n_boot=10000, verbose=0)
            d = helper.cohens_d_paired(np.array([group1, group2]),verbose=0)
            print(f'Proportion {proportion}, {ses}: p-value={p:.4f}, mean difference={meann:.2f}, 95% CI=({lowerr:.2f}, {upperr:.2f}), Cohen\'s d={d:.2f}')

            stars = determine_symbol(p)
            x_pos = i
            ax.text(x_pos, 1.1, stars, ha='center', va='top', fontsize=10)
            # draw a line between the two bars
            ax.plot([x_pos-0.2, x_pos+0.2], [1.05,1.05], color='black', linewidth=1)
            
    g.axhline(0, linestyle='--', linewidth=1, c='k') 
    g.set(xlabel='Proportion', ylabel='Max normalized density',title=f'{ses} session', ylim=[-0.1, 1.1])
    ax.legend_.remove()

fig.tight_layout()
sns.despine()
plt.savefig('results/plots/simulations_normalized_density.pdf', bbox_inches='tight', format='pdf', transparent=True)
